# The failure line, checked on validation (card #94)

cristiano's notebook for card #94, part of #33. The straight line from P(predecessor) to the failure rate is fitted on the training companies (formed January to September 2021). Does it predict how often the validation companies (formed October to December 2021) really fail?

**The labels.** The main label is the stricter one (decision note A strike-off counts only after real accounts, #175): a company failed if, within 24 months, it went into liquidation or administration, or it was struck off after filing at least one set of accounts that were not dormant. `real_slice.slice_from_raw` gives it. As the decision note asks, the old label (every strike-off within 24 months counts) is shown next to it, and insolvency only (liquidation or administration) comes last.

It runs the steps of `jonas_real.ipynb` (the labels, blocking, the matcher, P(predecessor), the line) on the whole crawl instead of a sample, with the same seed and the same costs, and changes nothing to make the result look better.

**The test set stays closed.** The 2022 companies are only counted: their own files (profile, insolvency, officers) are dropped before the adapter reads anything, so neither the labels nor the matcher see them, and the matcher learns only from records at companies formed by September 2021.

**Masking:** the notebook prints counts and rates only, never the name of a person or a company.

This project is a prototype and is not financial advice.

In [1]:
# Setup: pick up changes to src/ without restarting, and make `import src` work from notebooks/
%load_ext autoreload
%autoreload 2
import sys
sys.path.insert(0, "..")

import random
import time
import warnings
from pathlib import PurePosixPath

import numpy as np
import pandas as pd

from src import decide, evaluate, real_slice, resolve
from src import ingest_companies_house as adapter

SEED = 42              # one seed for everything random, as in main.ipynb and jonas_real.ipynb
random.seed(SEED)
np.random.seed(SEED)
COSTS = (10, 2, 1)     # the team's costs: a missed case, a wrong flag, asking

## The labels
The crawl's 2021 query companies go through the team's adapter, and `slice_from_raw` labels them with the main label; the 2022 companies' own files are dropped first, so they are only counted.

In [2]:
raw = adapter.load_raw()   # every file in data/raw, as real_slice.load_slice reads them
done = real_slice.query_companies()
numbers = list(done.loc[done["year"] == "2021", "company_number"])
test_numbers = set(done.loc[done["year"] == "2022", "company_number"])
assert not test_numbers & set(numbers) and len(numbers) + len(test_numbers) == len(done)
print(f"query companies: {len(done)} | 2021: {len(numbers)} | 2022 (test, counted only): {len(test_numbers)}")

# The 2022 companies' own files (profile, insolvency, officers) go before the adapter reads anything
raw = {path: data for path, data in raw.items()
       if not (PurePosixPath(path).parent.name in ("company", "insolvency", "officers")
               and PurePosixPath(path).stem in test_numbers)}
with warnings.catch_warnings(record=True) as caught:   # the adapter's warnings, without the company numbers in them
    warnings.simplefilter("always")
    mentions, claims, labels, left_out = real_slice.slice_from_raw(raw, numbers)
for warning in caught:
    print("adapter warning:", str(warning.message).split(":")[0])
print("left out (no label):", pd.Series(left_out).value_counts().to_dict() if left_out else "none")
print("labelled 2021 query companies by split:", labels["split"].value_counts(dropna=False).to_dict())
print("mentions:", mentions["kind"].value_counts().to_dict(), "| claims:", len(claims))

query companies: 13761 | 2021: 6914 | 2022 (test, counted only): 6847


adapter warning: 6 companies have has_insolvency_history but no insolvency file
adapter warning: 27 director records skipped
left out (no label): none
labelled 2021 query companies by split: {'train': 5362, 'validation': 1552}
mentions: {'company': 102015, 'person': 65806} | claims: 65806


How many companies failed under each label: the main label from `slice_from_raw`, the old label and insolvency only from `evaluate.label_failures`. The code checks that every insolvency-only failure is a main-label failure, and every main-label failure an old-label one.

In [3]:
labels = labels[labels["split"].isin(["train", "validation"])].reset_index(drop=True)
# failed: the main label, from slice_from_raw. failed_old: every strike-off within 24 months counts
# (label_failures' default). failed_insolvency: liquidation or administration only (count_dissolved=False).
labels["failed_old"] = evaluate.label_failures(mentions).loc[labels["company_id"]].to_numpy()
labels["failed_insolvency"] = evaluate.label_failures(mentions, count_dissolved=False).loc[labels["company_id"]].to_numpy()
assert not (labels["failed_insolvency"] & ~labels["failed"]).any()   # an insolvency always counts in the main label
assert not (labels["failed"] & ~labels["failed_old"]).any()          # a main-label failure is an old-label failure
train = labels[labels["split"] == "train"]
validation = labels[labels["split"] == "validation"]

LABELS = {"failed": "main label (stricter)", "failed_old": "old label (every strike-off)",
          "failed_insolvency": "insolvency only"}
pd.DataFrame({label: [f"{int(part[column].sum())} of {len(part)} ({part[column].mean():.1%})"
                      for part in (train, validation)]
              for column, label in LABELS.items()}, index=["train", "validation"])

,main label (stricter),old label (every strike-off),insolvency only
train,77 of 5362 (1.4%),1778 of 5362 (33.2%),15 of 5362 (0.3%)
validation,23 of 1552 (1.5%),527 of 1552 (34.0%),1 of 1552 (0.1%)


## P(predecessor)
Blocking proposes the pairs of person records worth comparing: the same surname and first initial, or the same birth month and year and a similar surname.

In [4]:
candidates = resolve.candidate_pairs(mentions)
print(f"{len(candidates)} candidate pairs from {int((mentions['kind'] == 'person').sum())} person records")

263601 candidate pairs from 65806 person records


The matcher learns from training pairs only, and only from pairs whose two records are at companies formed by September 2021. Then it gives every candidate pair its `p`, the chance that the two records are one person.

In [5]:
splits = real_slice.person_split(mentions, claims, labels)   # each person record's split, from the company it directs
labelled = resolve.labelled_pairs(mentions)                  # same officer id: label 1; a candidate pair with different ids: 0
labelled["split"] = real_slice.pair_split(labelled, splits)
later = (labelled["split"] == "train") & ~real_slice.train_period_pairs(labelled, mentions, claims)
labelled.loc[later, "split"] = "left out"                    # a record at a company formed after September 2021
train_pairs = labelled[labelled["split"] == "train"].reset_index(drop=True)
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    model = resolve.train_matcher(resolve.pair_features(train_pairs, mentions), train_pairs["label"], SEED)
for warning in caught:
    print("matcher warning:", str(warning.message)[:200])
print("training pairs by label:", train_pairs["label"].value_counts().sort_index().to_dict())
print("matcher weights:", resolve.matcher_weights(model).round(2).to_dict())

scored = resolve.scored_pairs(model, candidates, mentions)   # every candidate pair, with the matcher's p

training pairs by label: {0: 145518, 1: 102213}
matcher weights: {'name_similarity': 17.88, 'name_missing': 0.0, 'birth_same': 6.16, 'birth_missing': 0.13, 'postcode_same': 1.85, 'postcode_missing': 1.2, 'nationality_same': 1.94, 'nationality_missing': -0.08, 'intercept': -24.94}


Each company's matches are the pairs that tie one of its directors on its formation date to someone who ran a company insolvent by then (`decide.candidate_matches`), and P(predecessor) = 1 − ∏(1 − p) over them. `matches_for` hands `candidate_matches` only the rows that can change its answer, because the full tables make every call much slower: compare the time printed here with the one the check prints.

In [6]:
# A match is a pair from a director record of the company to a record at an insolvent company. So for one
# company only these rows can change candidate_matches' answer: the pairs that touch one of its director
# records, the claims at the company plus the claims of those pairs' records (each person record has
# exactly one claim), and the companies those claims point at.
start = time.perf_counter()
companies = mentions[mentions["kind"] == "company"]
assert companies["company_number"].is_unique and claims["subject"].is_unique
mention_of = companies.set_index("company_number")["mention_id"]
records_at = claims.groupby("object")["subject"].apply(list)   # a company's person records, any date
rows_of = {}                                                    # a person record's rows in `scored`
for row, (a, b) in enumerate(zip(scored["a"], scored["b"])):
    rows_of.setdefault(a, []).append(row)
    rows_of.setdefault(b, []).append(row)


def matches_for(query):
    """decide.candidate_matches(query, mentions, claims, scored), given only the rows that matter."""
    mention = mention_of[query]
    rows = sorted({row for record in records_at.get(mention, []) for row in rows_of.get(record, [])})
    pairs = scored.iloc[rows]                                   # same order as in `scored`
    ends = set(pairs["a"]) | set(pairs["b"])
    some_claims = claims[(claims["object"] == mention) | claims["subject"].isin(ends)]
    some_companies = companies[companies["mention_id"].isin(set(some_claims["object"]) | {mention})]
    return decide.candidate_matches(query, some_companies, some_claims, pairs)


matches = {query: matches_for(query) for query in labels["company_id"]}
p_pred = pd.Series({query: decide.company_p_predecessor(found) for query, found in matches.items()}, dtype=float)
print(f"matches and P(predecessor) for all {len(matches)} companies in {(time.perf_counter() - start) / 60:.1f} minutes")

matches and P(predecessor) for all 6914 companies in 2.2 minutes


The check: for 30 random companies and 30 with P(predecessor) above 0, the full call must give exactly the same matches as `matches_for`.

In [7]:
rng = np.random.default_rng(SEED)
positive = p_pred.index[p_pred > 0]
checked = [str(query) for query in list(rng.choice(labels["company_id"], 30, replace=False))
           + list(rng.choice(positive, min(30, len(positive)), replace=False))]
start = time.perf_counter()
for query in checked:
    full = decide.candidate_matches(query, mentions, claims, scored)
    assert (full is None) == (matches[query] is None), query
    if full is not None:
        pd.testing.assert_frame_equal(full, matches[query])
per_call = (time.perf_counter() - start) / len(checked)
print(f"check: matches_for equals candidate_matches on the full tables for all {len(checked)} companies checked "
      f"({sum(matches[q] is not None and not matches[q].empty for q in checked)} of them with at least one match)")
print(f"the full call takes {per_call:.2f} s a company: about {per_call * len(labels) / 60:.0f} minutes for all {len(labels)}")

check: matches_for equals candidate_matches on the full tables for all 60 companies checked (42 of them with at least one match)
the full call takes 0.24 s a company: about 28 minutes for all 6914


How many companies have a likely insolvent predecessor, in each split.

In [8]:
for name, part in [("train", train), ("validation", validation)]:
    p = p_pred.loc[part["company_id"]]
    print(f"{name}: = 0: {int((p == 0).sum())} | > 0: {int((p > 0).sum())} | >= 0.5: {int((p >= 0.5).sum())} | "
          f"no director records: {int(p.isna().sum())} | of {len(p)}")

train: = 0: 3511 | > 0: 1835 | >= 0.5: 351 | no director records: 16 | of 5362
validation: = 0: 1065 | > 0: 486 | >= 0.5: 105 | no director records: 1 | of 1552


## The line on train, and the thresholds
The thresholds the costs give. Then, for the main label and for the old label next to it: the straight line from P(predecessor) to the failure rate, fitted on the training companies (`decide.fit_failure_line`); what the decision layer can answer with it; and its prediction next to the real failure rate in three groups of P(predecessor), on the companies it was fitted on.

In [9]:
clear_below, flag_above = decide.thresholds(*COSTS)
print(f"costs {COSTS}: CLEAR at or below {clear_below:.0%}, FLAG at or above {flag_above:.0%}")


def fit_line(column):
    """decide.fit_failure_line on the training companies for one label, printed with its slope before
    clipping and with what the decision layer can answer on it."""
    x = p_pred.loc[train["company_id"]]
    line = decide.fit_failure_line(x, train[column])
    keep = x.notna().to_numpy()
    slope, intercept = np.polyfit(x.to_numpy()[keep], train[column].to_numpy()[keep].astype(float), 1)   # before clipping
    print(f"\n{LABELS[column]}: line fitted on {int(keep.sum())} training companies "
          f"({int(train[column][keep].sum())} positives): rate_without = {line[0]:.1%}, rate_with = {line[1]:.1%} | "
          f"slope {slope:+.4f} ({'down' if slope < 0 else 'up' if slope > 0 else 'flat'}), "
          f"intercept {intercept:.4f} before clipping")
    ends = {decide.verdict(rate, COSTS) for rate in line}   # every P(fail) on the line lies between its two ends
    print(f"P(fail) on this line runs from {min(line):.1%} to {max(line):.1%}: "
          + (f"{ends.pop()} for every company with director records" if len(ends) == 1
             else "the decision depends on P(predecessor)"))
    return line


def by_group(part, column, line):
    """The line's prediction next to the real failure rate, in three groups of P(predecessor) (jonas_real.ipynb)."""
    check = pd.DataFrame({"p_predecessor": p_pred.loc[part["company_id"]].to_numpy(),
                          "failed": part[column].to_numpy()})
    check["predicted"] = [decide.p_fail(p, line) for p in check["p_predecessor"]]
    groups = pd.cut(check["p_predecessor"], [-0.01, 0, 0.5, 1], labels=["0", "above 0, up to 0.5", "above 0.5"])
    table = check.groupby(groups, observed=True).agg(companies=("failed", "size"), failed=("failed", "sum"),
                                                     predicted=("predicted", "mean"))
    table["predicted"] = table["predicted"].map("{:.1%}".format)
    table["really failed (95% Wilson interval)"] = [
        f"{k / n:.1%} ({low:.1%} to {high:.1%})"
        for k, n, (low, high) in ((int(k), int(n), real_slice.wilson(int(k), int(n)))
                                  for k, n in zip(table["failed"], table["companies"]))]
    known = check["p_predecessor"].notna()
    print(f"all {int(known.sum())} with director records: line predicts {check.loc[known, 'predicted'].mean():.1%} "
          f"on average, really failed {check.loc[known, 'failed'].mean():.1%}")
    return table


line = fit_line("failed")            # the main label
display(by_group(train, "failed", line))
line_old = fit_line("failed_old")    # the old label, next to it
display(by_group(train, "failed_old", line_old))

costs (10, 2, 1): CLEAR at or below 10%, FLAG at or above 50%

main label (stricter): line fitted on 5346 training companies (77 positives): rate_without = 1.4%, rate_with = 1.8% | slope +0.0043 (up), intercept 0.0142 before clipping
P(fail) on this line runs from 1.4% to 1.8%: CLEAR for every company with director records
all 5346 with director records: line predicts 1.4% on average, really failed 1.4%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,3511,46,1.4%,1.3% (1.0% to 1.7%)
"above 0, up to 0.5",1484,25,1.4%,1.7% (1.1% to 2.5%)
above 0.5,351,6,1.8%,1.7% (0.8% to 3.7%)



old label (every strike-off): line fitted on 5346 training companies (1762 positives): rate_without = 34.1%, rate_with = 14.4% | slope -0.1967 (down), intercept 0.3411 before clipping
P(fail) on this line runs from 14.4% to 34.1%: NOT ENOUGH EVIDENCE for every company with director records
all 5346 with director records: line predicts 33.0% on average, really failed 33.0%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,3511,1235,34.1%,35.2% (33.6% to 36.8%)
"above 0, up to 0.5",1484,466,34.0%,31.4% (29.1% to 33.8%)
above 0.5,351,61,17.1%,17.4% (13.8% to 21.7%)


## The check on validation (#94)
The same tables for the validation companies, which the lines never saw: the main label first, the old label next to it. Where the prediction lies outside the 95% interval, the line is wrong for that group.

In [10]:
for column, this_line in [("failed", line), ("failed_old", line_old)]:
    print(f"-- {LABELS[column]}:")
    display(by_group(validation, column, this_line))

-- main label (stricter):
all 1551 with director records: line predicts 1.4% on average, really failed 1.5%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,1065,18,1.4%,1.7% (1.1% to 2.7%)
"above 0, up to 0.5",381,5,1.4%,1.3% (0.6% to 3.0%)
above 0.5,105,0,1.8%,0.0% (0.0% to 3.5%)


-- old label (every strike-off):
all 1551 with director records: line predicts 32.9% on average, really failed 33.9%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,1065,407,34.1%,38.2% (35.3% to 41.2%)
"above 0, up to 0.5",381,105,34.0%,27.6% (23.3% to 32.3%)
above 0.5,105,14,17.1%,13.3% (8.1% to 21.1%)


## The insolvency-only label
The same line and tables when only liquidation or administration counts as failing, the team's first label.

In [11]:
line_insolvency = fit_line("failed_insolvency")
for name, part in [("train", train), ("validation", validation)]:
    print(f"-- {name}:")
    display(by_group(part, "failed_insolvency", line_insolvency))


insolvency only: line fitted on 5346 training companies (15 positives): rate_without = 0.3%, rate_with = 0.3% | slope +0.0005 (up), intercept 0.0028 before clipping
P(fail) on this line runs from 0.3% to 0.3%: CLEAR for every company with director records
-- train:
all 5346 with director records: line predicts 0.3% on average, really failed 0.3%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,3511,10,0.3%,0.3% (0.2% to 0.5%)
"above 0, up to 0.5",1484,4,0.3%,0.3% (0.1% to 0.7%)
above 0.5,351,1,0.3%,0.3% (0.1% to 1.6%)


-- validation:
all 1551 with director records: line predicts 0.3% on average, really failed 0.1%


,companies,failed,predicted,really failed (95% Wilson interval)
p_predecessor,,,,
0,1065,1,0.3%,0.1% (0.0% to 0.5%)
"above 0, up to 0.5",381,0,0.3%,0.0% (0.0% to 1.0%)
above 0.5,105,0,0.3%,0.0% (0.0% to 3.5%)


**What it shows (AI draft, the owner puts it in their own words)**

- **Main label: the line is nearly flat and far under CLEAR.** Fitted on train, it goes from 1.4% with no likely insolvent predecessor to 1.8% with a certain one. Both ends are under the CLEAR threshold of 10%, so the decision layer answers CLEAR for every company with director records and FLAG never fires; a company with no director records still gets NOT ENOUGH EVIDENCE. On failure alone it does what "flag nothing" does.
- **On validation the main line fits, but it says almost nothing.** In the #94 table every group's prediction lies inside its 95% interval, and on average the line predicts 1.4% where 1.5% failed. But the three predictions differ by less than a point, the three intervals overlap, and the group above 0.5 has no failure at all: a likely insolvent predecessor makes no difference we can measure here.
- **Old label (every strike-off counts): the line slopes down**, from 34.1% to 14.4%, so every P(fail) lies between CLEAR and FLAG and every company gets NOT ENOUGH EVIDENCE. On validation it misses the two lower groups: its prediction lies below the interval at 0 and above it in the middle group. The decision note gives the likely reason this label turns the signal upside down: it mostly counts companies that never filed accounts, and people who ran a company before may be more likely to file. That reason was not tested in this notebook.
- **Insolvency only has too few failures to say anything:** 15 training companies and 1 validation company (see the label counts). Its line is flat at 0.3%, so it would also CLEAR every company with director records, and 1 failure on validation can't check it.
- **This is the project's main limit.** Under the main label, a company with a likely insolvent predecessor fails about as often as one without, so the failure line can't separate companies and the decision layer can't flag anyone: it was built for a signal this data doesn't show. As the decision note says, the headline moves to the matcher. Reported as found: the labels, the costs and the line were not changed to make it look better.